In [3]:

S3_BASE_URL = "https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator"

TABLES = ["trips", "users", "events", "payments"]


In [8]:
import requests
import xml.etree.ElementTree as ET

url = "https://inzhenerka-public.s3.eu-west-1.amazonaws.com/"
params = {"list-type": "2", "prefix": "scooters_data_generator/"}

resp = requests.get(url, params=params)
root = ET.fromstring(resp.content)

ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
for contents in root.findall(".//s3:Contents/s3:Key", ns):
    print(contents.text)

scooters_data_generator/
scooters_data_generator/events.parquet
scooters_data_generator/mo.geojson
scooters_data_generator/payments.parquet
scooters_data_generator/pyproject.toml
scooters_data_generator/scooters_raw.sql
scooters_data_generator/trips.parquet
scooters_data_generator/users.parquet
scooters_data_generator/version.txt
scooters_data_generator/weather.json


In [9]:
import dlt
import duckdb
import pandas as pd
pipeline = dlt.pipeline(
    pipeline_name='s3',
    destination='duckdb',
    dataset_name='raw',
)

In [12]:
duck = duckdb.connect()

In [27]:
t1 = "https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/trips.parquet"
duck.execute(f"CREATE TABLE if not exists trips AS SELECT * FROM '{t1}'")
duck.execute("SELECT * FROM trips LIMIT 5").df()

,id,user_id,scooter_hw_id,started_at,finished_at,start_lat,start_lon,finish_lat,finish_lon,distance,price
0,1,1393,GT-GXLV2-99D4,2023-06-01 06:06:02+03:00,2023-06-01 06:24:49+03:00,55.760750,37.613998,55.771710,37.618098,1644.987,0
1,2,1987,XM-1S-56D8X,2023-06-01 06:27:24+03:00,2023-06-01 06:53:51+03:00,55.742681,37.651062,55.737762,37.601283,4430.513,0
2,3,981,SN-ES2-82XF4,2023-06-01 06:32:32+03:00,2023-06-01 07:05:32+03:00,55.761562,37.642959,55.749707,37.589230,4473.653,0
3,4,1629,SN-MAXG30-56D2G,2023-06-01 06:36:45+03:00,2023-06-01 07:00:30+03:00,55.743430,37.590626,55.735752,37.614239,2506.703,23750
4,5,602,SN-MAXG30-56D2G,2023-06-01 06:42:08+03:00,2023-06-01 07:10:24+03:00,55.731653,37.623907,55.769428,37.623847,4883.543,0


In [39]:
t2 = "https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/weather.json"
duck.execute(f"CREATE TABLE if not exists weather AS SELECT * FROM '{t2}'")
duck.execute("SELECT * FROM weather").df()

,date,condition
0,2023-06-01T00:00:00.000,sun
1,2023-06-02T00:00:00.000,clouds
2,2023-06-03T00:00:00.000,sun
3,2023-06-04T00:00:00.000,sun
4,2023-06-05T00:00:00.000,sun
...,...,...
87,2023-08-27T00:00:00.000,sun
88,2023-08-28T00:00:00.000,sun
89,2023-08-29T00:00:00.000,sun
90,2023-08-30T00:00:00.000,clouds


In [31]:
duck.execute("INSTALL spatial; LOAD spatial;")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [36]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

In [38]:
t3 = "https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/mo.geojson"
duck.execute(f"CREATE TABLE IF NOT EXISTS mo AS SELECT *, geom::text as geomt FROM ST_Read('{t3}')")
duck.execute("SELECT * FROM mo LIMIT 5").df()

,NAME,OKATO,OKTMO,NAME_AO,OKATO_AO,ABBREV_AO,TYPE_MO,geom
0,Киевский,45298555,45945000,Троицкий,45298000,Троицкий,Поселение,"[5, 4, 0, 0, 0, 0, 0, 0, 95, 54, 19, 66, 189, 62, 93, 66, 65, 11, 20, 66, 182, 210, 93, 66, 5, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 34, 1, 0, 0, 0, 0, 0, 0, 116, 181, 21, 251, 203, 102, 66, 64, 63, 140, 16, 30, 109, 184, 75, 64, 180, 60, 15, 238, 206, 102, 66, 64, 241, 244, 74, 89, 134, 184, 75, 64, 251, 5, 187, 97, 219, 102, 66, 64, 25, 28, 37, 175, 206, 185, 75, 64, 107, 130, 168, 251, ...]"
1,Филёвский Парк,45268595,45328000,Западный,45268000,ЗАО,Муниципальный округ,"[2, 4, 0, 0, 0, 0, 0, 0, 233, 181, 21, 66, 154, 238, 94, 66, 16, 23, 22, 66, 161, 18, 95, 66, 2, 0, 0, 0, 1, 0, 0, 0, 113, 0, 0, 0, 0, 0, 0, 0, 52, 17, 54, 60, 189, 182, 66, 64, 36, 69, 100, 88, 197, 223, 75, 64, 222, 84, 164, 194, 216, 182, 66, 64, 164, 112, 61, 10, 215, 223, 75, 64, 2, 130, 57, 122, 252, 182, 66, 64, 135, 249, 242, 2, 236, 223, 75, 64, 229, 39, 213, 62, 29, 183, 66, 64, 171, 236, 187, 34, ...]"
2,Новофёдоровское,45298567,45954000,Троицкий,45298000,Троицкий,Поселение,"[2, 4, 0, 0, 0, 0, 0, 0, 219, 54, 19, 66, 247, 112, 93, 66, 212, 115, 20, 66, 211, 15, 94, 66, 2, 0, 0, 0, 1, 0, 0, 0, 163, 1, 0, 0, 0, 0, 0, 0, 251, 5, 187, 97, 219, 102, 66, 64, 25, 28, 37, 175, 206, 185, 75, 64, 9, 167, 5, 47, 250, 102, 66, 64, 1, 251, 232, 212, 149, 187, 75, 64, 163, 175, 32, 205, 88, 104, 66, 64, 100, 117, 171, 231, 164, 187, 75, 64, 35, 219, 249, 126, 106, 104, 66, 64, 6, 47, 250, 10, ...]"
3,Роговское,45298575,45956000,Троицкий,45298000,Троицкий,Поселение,"[2, 4, 0, 0, 0, 0, 0, 0, 187, 191, 19, 66, 149, 145, 92, 66, 56, 207, 20, 66, 43, 67, 93, 66, 2, 0, 0, 0, 1, 0, 0, 0, 102, 1, 0, 0, 0, 0, 0, 0, 100, 64, 246, 122, 247, 119, 66, 64, 109, 202, 21, 222, 229, 158, 75, 64, 171, 236, 187, 34, 248, 119, 66, 64, 123, 49, 148, 19, 237, 158, 75, 64, 178, 46, 110, 163, 1, 120, 66, 64, 52, 162, 180, 55, 248, 158, 75, 64, 28, 206, 252, 106, 14, 120, 66, 64, 94, 186, 73, 12, ...]"
4,"""Мосрентген""",45297568,45953000,Новомосковский,45297000,Новомосковский,Поселение,"[2, 4, 0, 0, 0, 0, 0, 0, 28, 194, 21, 66, 160, 102, 94, 66, 231, 244, 21, 66, 176, 140, 94, 66, 2, 0, 0, 0, 1, 0, 0, 0, 108, 0, 0, 0, 0, 0, 0, 0, 156, 80, 136, 128, 67, 184, 66, 64, 21, 58, 175, 177, 75, 208, 75, 64, 56, 45, 120, 209, 87, 184, 66, 64, 70, 95, 65, 154, 177, 208, 75, 64, 234, 120, 204, 64, 101, 184, 66, 64, 155, 143, 107, 67, 197, 208, 75, 64, 42, 29, 172, 255, 115, 184, 66, 64, 240, 162, 175, 32, ...]"


In [15]:
duck.execute(f"SELECT * FROM '{t1}'").fetch_arrow_table()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pyarrow.Table
id: int64
user_id: int64
scooter_hw_id: string
started_at: timestamp[us, tz=Europe/Moscow]
finished_at: timestamp[us, tz=Europe/Moscow]
start_lat: double
start_lon: double
finish_lat: double
finish_lon: double
distance: double
price: int64
----
id: [[1,2,3,4,5,...,105897,105898,105899,105900,105901]]
user_id: [[1393,1987,981,1629,602,...,493,490,1166,235,1190]]
scooter_hw_id: [["GT-GXLV2-99D4","XM-1S-56D8X","SN-ES2-82XF4","SN-MAXG30-56D2G","SN-MAXG30-56D2G",...,"GT-GXLV2-99D4","SN-AIRT15-84JK9","GT-GXLV2-99D4","SN-MAXG30-56D2G","GT-GXLV2-99D4"]]
started_at: [[2023-06-01 03:06:02.000000Z,2023-06-01 03:27:24.000000Z,2023-06-01 03:32:32.000000Z,2023-06-01 03:36:45.000000Z,2023-06-01 03:42:08.000000Z,...,2023-08-30 18:40:15.000000Z,2023-08-30 18:44:26.000000Z,2023-08-30 18:46:36.000000Z,2023-08-30 18:47:56.000000Z,2023-08-30 19:52:19.000000Z]]
finished_at: [[2023-06-01 03:24:49.000000Z,2023-06-01 03:53:51.000000Z,2023-06-01 04:05:32.000000Z,2023-06-01 04:00:30.000000Z,2023-06